# Digital Cash Management Forecast: lab (data scientist)

Try other models here without touching production: nothing in this notebook changes the flow, the saved models or
the production model.

- **Same test as production.** Your forecaster is trained without the last 6 months before the cut-off and tested
  on them, the way champion / challenger tests a new model. Your scores sit next to the production model's and the
  simple guess (the average of the last 3 months), with the decision champion / challenger would take.
- **Each try is recorded** in Experiment Tracking, experiment `cashflow_lab`: its settings, its test scores and the
  trained forecaster. Compare tries there (Lab → Experiment tracking). Lab tries are never deployed.
- **The same try without a notebook:** set the project variable `lab_forecaster` to a forecaster's name and build
  `lab_results` (Lab zone). Each try adds its rows to `lab_results`.

**Kernel:** the code env `cashflow_forecast` (Kernel → Change kernel). If it is not in the list, the env has no
Jupyter support yet: ask whoever manages it to enable it, and meanwhile use `lab_results`.

**A forecaster** is a class with `name`, `params` (its settings), `inputs` (the model-row columns it reads),
`fit(rows)` and `predict(rows)`, which returns `p10`, `ev` and `p90` in SAR (details in `models.py`, project
library). The amount to predict is the column `target`: the month's amount minus the obligations already known at
the cut-off, which the forecast adds back.

In [ ]:
import importlib

try:
    import catboost
    import mlflow
except ImportError:
    raise RuntimeError("Use the kernel of the code env cashflow_forecast (Kernel > Change kernel). If it is not in the "
                       "list, ask whoever manages that code env to enable its Jupyter support; meanwhile, set the "
                       "project variable lab_forecaster and build lab_results (Lab zone).")
import cashflow
import models
import tracking

for module in (cashflow, models, tracking):     # pick up changes to the project library without a restart
    importlib.reload(module)
from models import CatBoostForecaster, lab_try

In [ ]:
# Your forecaster: change anything (its settings, its inputs, or fit and predict written from scratch). To try one
# from forecasters.py instead: import forecasters, importlib.reload(forecasters), then use its class below.
class MyForecaster(CatBoostForecaster):
    """Example: the reference CatBoost forecaster with deeper trees."""
    name = "my_forecaster"
    params = {**CatBoostForecaster.params, "depth": 8}


table, decision, reason, run_id = lab_try(MyForecaster())
print("EV error (lower is better) and P10-P90 coverage (aim: 80%), per category, all months ahead together:")
display(table)
print(f"Champion / challenger would decide: {decision} ({reason})")
print("Recorded in Experiment Tracking, experiment cashflow_lab, run", run_id)

## Put it in production

1. Copy the class into `forecasters.py` (project library) and add it to `FORECASTERS`. The setup notebook never
   overwrites that file.
2. Set the project variable `forecaster` to its name.
3. Run the scenario **Retrain, test and promote**. Champion / challenger promotes it only if it beats the
   production model on the same 6 months; the next Nightly batch forecasts with it.
4. To stop training it, set `forecaster` back to `catboost`. To use an earlier model now, activate its version in
   the three saved models.